# Tutorial 6: KKT Conditions

**Prerequisites:** Tutorial 1 — What Is an Optimization Problem?, Tutorial 2 — Types of Optimization Problems, Tutorial 3 — Gradient Descent from Scratch, Tutorial 5 — Newton and Quasi-Newton Methods  
**Up Next:** Tutorial 7 — Sequential Quadratic Programming (SQP)

---

## Concept

So far, we have minimized the four-bar objective without any constraints — the optimizer was free to pick any link lengths. In practice, mechanisms must satisfy physical requirements. A classic one is the **Grashof condition**: the shortest link plus the longest must be no greater than the sum of the other two, otherwise the mechanism cannot make a full revolution.

The **Karush-Kuhn-Tucker (KKT) conditions** generalize the familiar "set the gradient to zero" rule to constrained problems. They introduce **Lagrange multipliers** $\lambda_i$ — one per constraint — that measure how much each constraint "costs" at the optimum.

## Key Takeaway

> **At a constrained optimum, the KKT conditions must hold: stationarity (gradient of Lagrangian is zero), primal feasibility (constraints satisfied), dual feasibility ($\lambda \ge 0$), and complementary slackness ($\lambda_i \, g_i = 0$). Active constraints shape the solution; inactive ones are irrelevant.**

## Math Core

For the problem $\min f(\mathbf{x})$ subject to $g_i(\mathbf{x}) \le 0$, the **Lagrangian** is:

$$\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}) = f(\mathbf{x}) + \sum_i \lambda_i \, g_i(\mathbf{x})$$

The KKT conditions at an optimum $(\mathbf{x}^*, \boldsymbol{\lambda}^*)$ are:

| Condition | Formula |
|---|---|
| **Stationarity** | $\nabla_\mathbf{x} \mathcal{L} = \nabla f + \sum_i \lambda_i \nabla g_i = 0$ |
| **Primal feasibility** | $g_i(\mathbf{x}^*) \le 0$ for all $i$ |
| **Dual feasibility** | $\lambda_i \ge 0$ for all $i$ |
| **Complementary slackness** | $\lambda_i \, g_i(\mathbf{x}^*) = 0$ for all $i$ |

A constraint is **active** when $g_i(\mathbf{x}^*) = 0$ (binding at the optimum) and **inactive** when $g_i(\mathbf{x}^*) < 0$ (slack). Complementary slackness says inactive constraints have $\lambda_i = 0$.

## Code

We define the Grashof constraint inline and compare unconstrained vs. constrained solutions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
from scipy.optimize import minimize
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### The Grashof constraint

A four-bar linkage satisfies the Grashof condition when the sum of the shortest and longest links is at most the sum of the other two. We write this as $g(\mathbf{x}) \le 0$:

$$g(\mathbf{x}) = (\ell_{\max} + \ell_{\min}) - (\ell_{\text{sum}} - \ell_{\max} - \ell_{\min}) \le 0$$

In [ ]:
def grashof_g(x):
    """Grashof constraint: g(x) <= 0 means Grashof is satisfied."""
    lengths = np.array([L0, L1, x[0], x[1]])
    s = np.sum(lengths)
    return (np.max(lengths) + np.min(lengths)) - (s - np.max(lengths) - np.min(lengths))

# Test: equal links always satisfy Grashof
print(f'g([1.0, 1.0]) = {grashof_g([1.0, 1.0]):.4f}  (= 0, boundary)')
print(f'g([0.5, 2.5]) = {grashof_g([0.5, 2.5]):.4f}  (> 0, violated)')
print(f'g([1.5, 1.5]) = {grashof_g([1.5, 1.5]):.4f}  (= 0, boundary)')

### Unconstrained vs. constrained optimum

Let's solve both problems and see how the Grashof constraint changes the solution.

In [ ]:
x0 = np.array([1.0, 1.5])
bnds = [(0.3, 3.0), (0.3, 3.0)]

# Unconstrained
res_unc = minimize(objective, x0, method='L-BFGS-B',
                   bounds=bnds, options={'ftol': 1e-12, 'eps': 1e-7})

# Constrained (scipy 'ineq' means c(x) >= 0)
res_con = minimize(objective, x0, method='SLSQP', bounds=bnds,
                   constraints={'type': 'ineq', 'fun': lambda x: -grashof_g(x)})

print('Unconstrained:')
print(f'  x* = [{res_unc.x[0]:.4f}, {res_unc.x[1]:.4f}]')
print(f'  f  = {res_unc.fun:.4f}')
print(f'  g  = {grashof_g(res_unc.x):.4f}  '
      f'{"(violated!)" if grashof_g(res_unc.x) > 1e-6 else "(feasible)"}')

print('\nConstrained (Grashof):')
print(f'  x* = [{res_con.x[0]:.4f}, {res_con.x[1]:.4f}]')
print(f'  f  = {res_con.fun:.4f}')
print(f'  g  = {grashof_g(res_con.x):.4f}  (active constraint)')

### Visualizing the feasible region

The Grashof-feasible region ($g \le 0$) is where the mechanism can make a full crank revolution. The unconstrained optimum may lie outside this region.

In [ ]:
l2v = np.linspace(0.3, 2.5, 80)
l3v = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)
G = np.vectorize(lambda a, b: grashof_g([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
# Shade infeasible region
ax.contourf(L2, L3, G, levels=[0, 10], colors=['red'], alpha=0.25)
ax.contour(L2, L3, G, levels=[0], colors=['red'], linewidths=2)
# Mark optima
ax.plot(*res_unc.x, 'wo', ms=10, markeredgecolor='k',
        label=f'unconstrained (f={res_unc.fun:.3f})')
ax.plot(*res_con.x, 'r*', ms=14, markeredgecolor='k',
        label=f'constrained (f={res_con.fun:.3f})')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Feasible region (white) and Grashof boundary (red line)')
ax.legend(loc='upper left')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Active vs. inactive constraints

At the constrained optimum, the Grashof constraint is **active** ($g \approx 0$), meaning it directly shapes the solution. If the unconstrained optimum happened to already satisfy Grashof, the constraint would be **inactive** ($g < 0$, $\lambda = 0$) and the two solutions would coincide.

The non-zero Lagrange multiplier $\lambda > 0$ at an active constraint tells us how much the objective would improve if we relaxed that constraint slightly. A larger $\lambda$ means the constraint is more "expensive."

## Mechanism Hook

Let's compare the two mechanisms side by side — the unconstrained optimum may produce a linkage that locks up during rotation, while the Grashof-feasible one guarantees full crank motion.

In [ ]:
from dms.mechanisms.fourbar import FourBar

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, xopt, title in [
    (axes[0], res_unc.x, 'Unconstrained'),
    (axes[1], res_con.x, 'Grashof-constrained'),
]:
    l2, l3 = xopt
    mech = FourBar(L0, L1, l2, l3)
    path = np.array([coupler_point(t, l2, l3) for t in THETAS])
    mech.plot(np.deg2rad(90), ax=ax)
    ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
    if path[0] is not None:
        ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.set_title(f'{title}\n$\\ell_2$={l2:.3f}, $\\ell_3$={l3:.3f}'
                 f'  |  f={objective(xopt):.3f}, g={grashof_g(xopt):.3f}')
plt.tight_layout()

The constrained solution has a slightly worse objective value but guarantees a physically viable mechanism. In Tutorial 7, we explore **SQP**, an algorithm that handles these constrained problems systematically by solving a sequence of quadratic subproblems.

---